# GeoDiff-GAN: Official SR Backbone + SN-HFR Paired Benchmark

This notebook trains an unchanged official x4 SR backbone, freezes it, then trains the shared Sensor-Nullspace High-Frequency Refiner (SN-HFR). Each model is compared only with its own refined version at its official native training crop. It does not construct a cross-model leaderboard.

Expected prepared data: NPZ patches containing an `hr` RGB array and, preferably, a JSONL manifest. Patches stay in `/kaggle/input`; the notebook writes only manifests, checkpoints, metrics, and diagnostics to `/kaggle/working`.

## 1. Settings

Start with `PROFILE="smoke"` and one model. After the complete notebook passes, change to `screening`. PFT is disabled by default because its official implementation requires compiling `smm_cuda`.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, re, time

REPOSITORY_URL = "https://github.com/shashankjs2002/SI-SR-1.git"
REPOSITORY_DIR = Path("/kaggle/working/geodiff-gan")
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/geodiff-sn-hfr")
SOURCE_ROOT = WORK_ROOT / "official_sources"
RUN_ROOT = WORK_ROOT / "paired_runs"
LOCAL_MANIFEST = WORK_ROOT / "manifest_native_pairing.jsonl"

PROFILE = "smoke"  # smoke | screening | paper
MODELS_TO_RUN = ["swinir"]
# Suggested staged set after smoke: swinir, omnisr, srformer, ttst, mfghmoe, sat
# Full registry is printed after installation. Run large models in separate sessions.
ENABLE_PFT = False
ENABLE_OPTIONAL_METRICS = False  # LPIPS/DISTS may download weights
FORCE_TILE_LEVEL_SPLITS = True
DATASET_ROOT_OVERRIDE = None  # e.g. Path('/kaggle/input/my-prepared-npz-dataset')
RESTORE_RUNS_FROM = None       # attached previous Kaggle output/dataset, if any
SEED = 42

WORK_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
print("Work root:", WORK_ROOT)
print("Models:", MODELS_TO_RUN)
try:
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as error:
    print("Torch check failed:", error)

## 2. Command helper and optional run restoration

In [ ]:
def run(command, cwd=None, check=True):
    command = [str(value) for value in command]
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    print("+", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, env=environment, check=check)

if RESTORE_RUNS_FROM is not None:
    restore = Path(RESTORE_RUNS_FROM)
    if not restore.exists():
        raise FileNotFoundError(restore)
    source_runs = restore / "paired_runs" if (restore / "paired_runs").is_dir() else restore
    for item in source_runs.iterdir():
        destination = RUN_ROOT / item.name
        if destination.exists():
            print("Keeping current run:", destination)
        elif item.is_dir():
            shutil.copytree(item, destination)
        else:
            shutil.copy2(item, destination)
    print("Restored available checkpoints without replacing current files.")

## 3. Clone and install GeoDiff-GAN

This cell never deletes the repository directory. A non-Git folder is accepted only when it already contains `src/geodiff_gan`.

In [ ]:
if REPOSITORY_DIR.exists():
    if (REPOSITORY_DIR / ".git").is_dir():
        print("Updating existing clone:", REPOSITORY_DIR)
        run(["git", "pull", "--ff-only"], cwd=REPOSITORY_DIR)
    elif (REPOSITORY_DIR / "src" / "geodiff_gan").is_dir():
        print("Using existing non-Git repository copy:", REPOSITORY_DIR)
    else:
        raise RuntimeError(f"Existing directory is not a usable repository: {REPOSITORY_DIR}")
else:
    run(["git", "clone", "--depth", "1", REPOSITORY_URL, REPOSITORY_DIR])

required_refiner = REPOSITORY_DIR / "src" / "geodiff_gan" / "benchmark" / "refiner.py"
if not required_refiner.exists():
    raise RuntimeError(
        "The cloned branch does not contain SN-HFR yet. Push the current local "
        "GeoDiff-GAN changes to REPOSITORY_URL, then restart this notebook."
    )

PIP = [sys.executable, "-m", "pip"]
run([*PIP, "install", "-q", "timm>=1.0.15", "einops>=0.8", "pandas>=2", "matplotlib>=3.8", "tqdm>=4.66"])
if ENABLE_OPTIONAL_METRICS:
    run([*PIP, "install", "-q", "lpips", "DISTS-pytorch"])
run([*PIP, "install", "-q", "-e", ".", "--no-deps"], cwd=REPOSITORY_DIR)
sys.path.insert(0, str(REPOSITORY_DIR / "src"))
os.chdir(REPOSITORY_DIR)

import geodiff_gan
from geodiff_gan.benchmark.models import MODEL_SPECS
print("Repository:", REPOSITORY_DIR)
print("Available official adapters:", sorted(MODEL_SPECS))

## 4. Locate NPZ patches, rebase the manifest, and enforce tile splits

No NPZ file is copied. If no usable manifest is attached, one is built from the NPZ paths. Complete inferred tiles are assigned to one split to avoid patch leakage.

In [ ]:
from collections import Counter, defaultdict
from geodiff_gan.data.manifest import ManifestRecord, write_manifest

DATA_ROOT = Path(DATASET_ROOT_OVERRIDE) if DATASET_ROOT_OVERRIDE else INPUT_ROOT
if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
npz_files = sorted(DATA_ROOT.rglob("*.npz"))
if not npz_files:
    raise FileNotFoundError(
        "No prepared NPZ patches were found. Attach the dataset produced by "
        "Sentinel2_HR_Prepare_and_Caption.ipynb; this benchmark intentionally "
        "does not expand SAFE products into limited Kaggle working storage."
    )
print(f"Found {len(npz_files):,} NPZ patches under {DATA_ROOT}")

suffix_index = defaultdict(list)
for path in npz_files:
    parts = path.as_posix().split("/")
    for depth in range(1, min(5, len(parts)) + 1):
        suffix_index["/".join(parts[-depth:])].append(path)

def resolve_patch(old_value):
    old_path = Path(str(old_value))
    if old_path.exists():
        return old_path.resolve()
    normalized = str(old_value).replace("\\", "/").strip("/")
    parts = normalized.split("/")
    for depth in range(min(5, len(parts)), 0, -1):
        matches = suffix_index.get("/".join(parts[-depth:]), [])
        if len(matches) == 1:
            return matches[0].resolve()
    raise FileNotFoundError(f"Could not uniquely rebase patch: {old_value}")

manifest_candidates = sorted(
    path for path in DATA_ROOT.rglob("*.jsonl") if "manifest" in path.name.lower()
)
source_values = None
source_manifest = None
for candidate in manifest_candidates:
    try:
        values = [json.loads(line) for line in candidate.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not values or "patch" not in values[0]:
            continue
        for value in values:
            resolve_patch(value["patch"])
        source_values = values
        source_manifest = candidate
        break
    except Exception as error:
        print("Skipping unusable manifest:", candidate, "->", error)

tile_pattern = re.compile(r"(?:^|[^A-Z0-9])T?(\d{2}[A-Z]{3})(?:[^A-Z0-9]|$)")
coordinate_pattern = re.compile(r"_r(\d+)_c(\d+)", re.IGNORECASE)

def infer_tile(path):
    for part in reversed(path.parts):
        match = tile_pattern.search(part.upper())
        if match:
            return match.group(1)
    return path.parent.name

def infer_coordinates(path):
    match = coordinate_pattern.search(path.stem)
    return (int(match.group(1)), int(match.group(2))) if match else (0, 0)

if source_values is None:
    source_values = []
    for path in npz_files:
        row, col = infer_coordinates(path)
        source_values.append({
            "patch": str(path.resolve()), "tile_id": infer_tile(path),
            "row": row, "col": col, "valid_fraction": 1.0,
            "source_product": path.parent.name,
        })
    print("No reusable manifest found; generated records from NPZ paths.")
else:
    print("Rebasing manifest:", source_manifest)

tiles = sorted({str(value.get("tile_id") or infer_tile(resolve_patch(value["patch"]))) for value in source_values})
if len(tiles) < 3:
    raise RuntimeError(f"At least three geographic tiles are required for train/val/test; found {tiles}")
ranked_tiles = sorted(tiles, key=lambda value: hashlib.sha256(f"{SEED}:{value}".encode()).hexdigest())
test_count = max(1, round(len(ranked_tiles) * 0.10))
val_count = max(1, round(len(ranked_tiles) * 0.10))
if test_count + val_count >= len(ranked_tiles):
    test_count, val_count = 1, 1
tile_split = {tile: "train" for tile in ranked_tiles}
for tile in ranked_tiles[:test_count]:
    tile_split[tile] = "test"
for tile in ranked_tiles[test_count:test_count + val_count]:
    tile_split[tile] = "val"

records = []
for value in source_values:
    patch = resolve_patch(value["patch"])
    tile_id = str(value.get("tile_id") or infer_tile(patch))
    row, col = infer_coordinates(patch)
    split = tile_split[tile_id] if FORCE_TILE_LEVEL_SPLITS else value.get("split", tile_split[tile_id])
    records.append(ManifestRecord(
        patch=str(patch), tile_id=tile_id, split=split,
        row=int(value.get("row", row)), col=int(value.get("col", col)),
        valid_fraction=float(value.get("valid_fraction", 1.0)),
        source=str(value.get("source", "copernicus_sentinel2_l2a")),
        license_id=str(value.get("license_id", "copernicus-free-full-open")),
        caption=str(value.get("caption", "")),
        source_product=str(value.get("source_product", patch.parent.name)),
    ))
write_manifest(LOCAL_MANIFEST, records)
print("Local manifest:", LOCAL_MANIFEST)
print("Patch splits:", Counter(record.split for record in records))
print("Tile splits:", Counter(tile_split.values()), tile_split)

## 5. Visualize five prepared samples

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.nn import functional as F
from geodiff_gan.data import SentinelPatchDataset

preview_dataset = SentinelPatchDataset(
    LOCAL_MANIFEST, split="train", scale=4, caption_file=None,
    augment=False, random_degradation=False, degradation_seed=SEED,
    degradation_severity="mild",
)
preview_indices = random.Random(SEED).sample(range(len(preview_dataset)), min(5, len(preview_dataset)))
figure, axes = plt.subplots(len(preview_indices), 3, figsize=(12, 4 * len(preview_indices)), squeeze=False)
for row, index in enumerate(preview_indices):
    sample = preview_dataset[index]
    lr, hr = sample["lr"], sample["hr"]
    bicubic = F.interpolate(lr[None], size=hr.shape[-2:], mode="bicubic", align_corners=False)[0].clamp(0, 1)
    for axis, image, title in zip(axes[row], [lr, bicubic, hr], [f"LR index {index}", "Bicubic", "Target HR"]):
        axis.imshow(image[:3].clamp(0, 1).permute(1, 2, 0))
        axis.set_title(title)
        axis.axis("off")
plt.tight_layout()
plt.show()

## 6. Clone only the selected official model sources

Internet must be enabled unless these repositories are attached separately. Existing source directories are kept.

In [ ]:
unknown = sorted(set(MODELS_TO_RUN) - set(MODEL_SPECS))
if unknown:
    raise ValueError(f"Unknown model keys: {unknown}")
if "pft" in MODELS_TO_RUN and not ENABLE_PFT:
    raise RuntimeError("Set ENABLE_PFT=True to compile the official PFT smm_cuda extension.")

for key in MODELS_TO_RUN:
    spec = MODEL_SPECS[key]
    destination = SOURCE_ROOT / spec.directory
    if destination.exists():
        print("Keeping existing source:", key, destination)
    else:
        run(["git", "clone", "--depth", "1", spec.repository, destination])

mamba_models = {"fremamba", "mambair", "mambairv2"} & set(MODELS_TO_RUN)
if mamba_models:
    run([*PIP, "install", "causal-conv1d>=1.4.0", "mamba-ssm>=2.2.0", "thop"])
if "pft" in MODELS_TO_RUN:
    run([*PIP, "install", "fairscale"])
    run(["bash", "make.sh"], cwd=SOURCE_ROOT / "PFT-SR" / "ops_smm")
print("Selected sources are ready.")

## 7. Native-size protocol and compute profile

In [ ]:
PROFILES = {
    "smoke": {
        "base_updates": 20, "refiner_updates": 20,
        "validate_every": 10, "validation_limit": 4, "test_limit": 4,
        "early_stopping": 2, "bootstrap": 100,
    },
    "screening": {
        "base_updates": 5000, "refiner_updates": 3000,
        "validate_every": 500, "validation_limit": 64, "test_limit": 80,
        "early_stopping": 5, "bootstrap": 1000,
    },
    "paper": {
        "base_updates": 50000, "refiner_updates": 15000,
        "validate_every": 1000, "validation_limit": 256, "test_limit": 400,
        "early_stopping": 8, "bootstrap": 2000,
    },
}
if PROFILE not in PROFILES:
    raise ValueError(PROFILE)
SETTINGS = PROFILES[PROFILE]
BASE_BATCH, BASE_ACCUMULATION = 1, 8
REFINER_BATCH, REFINER_ACCUMULATION = 1, 4
REFINER_CHANNELS = 32
REFINER_BLOCKS = (2, 2, 3)

print("Profile:", PROFILE, SETTINGS)
for key in MODELS_TO_RUN:
    spec = MODEL_SPECS[key]
    print(f"{key}: official {spec.native_lr_size}x{spec.native_lr_size} LR -> {spec.native_lr_size * spec.scale}x{spec.native_lr_size * spec.scale} HR")

## 8. Probe every unchanged official architecture

In [ ]:
PROBE_RESULTS = {}
for model_name in MODELS_TO_RUN:
    command = [
        sys.executable, "-m", "geodiff_gan.cli.benchmark_sota",
        "--model", model_name, "--source-root", SOURCE_ROOT,
        "--manifest", LOCAL_MANIFEST,
        "--output", RUN_ROOT / model_name / "base", "--probe-only",
    ]
    completed = subprocess.run([str(value) for value in command], cwd=REPOSITORY_DIR, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Official probe failed: {model_name}")
    PROBE_RESULTS[model_name] = json.loads(completed.stdout[completed.stdout.index("{"):])
print("All native-size x4 probes passed.")

## 9. Train official bases sequentially

Rerunning resumes each model from `latest.pt`. Only `best.pt` and `latest.pt` are retained.

In [ ]:
BASE_CHECKPOINTS = {}
for model_name in MODELS_TO_RUN:
    output = RUN_ROOT / model_name / "base"
    command = [
        sys.executable, "-m", "geodiff_gan.cli.benchmark_sota",
        "--model", model_name, "--source-root", SOURCE_ROOT,
        "--manifest", LOCAL_MANIFEST, "--output", output,
        "--architecture-mode", "official",
        "--max-updates", SETTINGS["base_updates"],
        "--batch-size", BASE_BATCH, "--accumulation", BASE_ACCUMULATION,
        "--learning-rate", 2e-4, "--weight-decay", 1e-4,
        "--num-workers", 2,
        "--validation-limit", SETTINGS["validation_limit"],
        "--test-limit", SETTINGS["test_limit"],
        "--validate-every", SETTINGS["validate_every"],
        "--early-stopping-patience", SETTINGS["early_stopping"],
        "--degradation-seed", SEED, "--degradation-severity", "mild",
        "--seed", SEED,
    ]
    print("\n" + "=" * 88, "\nOFFICIAL BASE:", model_name, "\n" + "=" * 88)
    run(command, cwd=REPOSITORY_DIR)
    checkpoint = output / "best.pt"
    if not checkpoint.exists():
        checkpoint = output / "latest.pt"
    if not checkpoint.exists():
        raise FileNotFoundError(f"No base checkpoint for {model_name}")
    BASE_CHECKPOINTS[model_name] = checkpoint
    print("Selected frozen base:", checkpoint)

## 10. Train and evaluate SN-HFR for each frozen base

The same 1.16M-parameter refiner is used for every compatible backbone. The exact base checkpoint is protected by SHA-256 in the refiner checkpoint.

In [ ]:
PAIRED_RESULTS = {}
for model_name in MODELS_TO_RUN:
    output = RUN_ROOT / model_name / "sn_hfr"
    command = [
        sys.executable, "-m", "geodiff_gan.cli.benchmark_refiner",
        "--mode", "train-evaluate", "--model", model_name,
        "--source-root", SOURCE_ROOT, "--manifest", LOCAL_MANIFEST,
        "--base-checkpoint", BASE_CHECKPOINTS[model_name], "--output", output,
        "--max-updates", SETTINGS["refiner_updates"],
        "--batch-size", REFINER_BATCH, "--accumulation", REFINER_ACCUMULATION,
        "--learning-rate", 1e-4, "--weight-decay", 1e-4,
        "--num-workers", 2,
        "--validation-limit", SETTINGS["validation_limit"],
        "--test-limit", SETTINGS["test_limit"],
        "--validate-every", SETTINGS["validate_every"],
        "--early-stopping-patience", SETTINGS["early_stopping"],
        "--channels", REFINER_CHANNELS,
        "--blocks-per-level", *REFINER_BLOCKS,
        "--max-residual", 0.12, "--nullspace-iterations", 1,
        "--nullspace-step", 0.75,
        "--save-images", 5, "--bootstrap-samples", SETTINGS["bootstrap"],
        "--degradation-seed", SEED, "--degradation-severity", "mild",
        "--seed", SEED,
    ]
    if ENABLE_OPTIONAL_METRICS:
        command.append("--optional-metrics")
    print("\n" + "=" * 88, "\nSN-HFR PAIR:", model_name, "\n" + "=" * 88)
    run(command, cwd=REPOSITORY_DIR)
    metrics_path = output / "test_paired_metrics.json"
    PAIRED_RESULTS[model_name] = json.loads(metrics_path.read_text(encoding="utf-8"))

## 11. Read paired results correctly

Every displayed table is a base-versus-refined pair for one model. Do not rank rows belonging to different models.

In [ ]:
import pandas as pd
from IPython.display import display

for model_name in MODELS_TO_RUN:
    result = PAIRED_RESULTS[model_name]
    metrics = sorted(result["base"])
    rows = []
    for metric in metrics:
        pair = result["paired"][metric]
        rows.append({
            "metric": metric,
            "direction": pair["direction"],
            "official_base": result["base"][metric],
            "base_plus_sn_hfr": result["refined"][metric],
            "delta_refined_minus_base": pair["mean_delta_refined_minus_base"],
            "ci_low": pair["bootstrap_95ci"][0],
            "ci_high": pair["bootstrap_95ci"][1],
            "improved_patch_fraction": pair["improved_patch_fraction"],
        })
    print(f"\n{model_name}: native LR {result['native_lr_size']} -> HR {result['native_hr_size']} (paired only)")
    display(pd.DataFrame(rows).round(6))

## 12. Training curves

In [ ]:
for model_name in MODELS_TO_RUN:
    base_history_path = RUN_ROOT / model_name / "base" / "history.jsonl"
    refiner_history_path = RUN_ROOT / model_name / "sn_hfr" / "history.jsonl"
    base_history = pd.DataFrame(json.loads(line) for line in base_history_path.read_text(encoding="utf-8").splitlines() if line.strip())
    refiner_history = pd.DataFrame(json.loads(line) for line in refiner_history_path.read_text(encoding="utf-8").splitlines() if line.strip())
    figure, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(base_history["update"], base_history["loss"], label="official base")
    axes[0].set_title("Base training loss")
    axes[1].plot(refiner_history["update"], refiner_history["loss"], label="SN-HFR", color="tab:orange")
    axes[1].set_title("Refiner training loss")
    validation = refiner_history.dropna(subset=["val_refined_psnr"]) if "val_refined_psnr" in refiner_history else refiner_history.iloc[:0]
    if len(validation):
        axes[2].plot(validation["update"], validation["val_base_psnr"], marker="o", label="frozen base")
        axes[2].plot(validation["update"], validation["val_refined_psnr"], marker="s", label="base + SN-HFR")
    axes[2].set_title("Paired validation PSNR")
    for axis in axes:
        axis.set_xlabel("optimizer update")
        axis.grid(alpha=0.2)
        axis.legend()
    figure.suptitle(model_name)
    plt.tight_layout()
    plt.show()

## 13. Inspect any saved paired sample

Change `VIEW_MODEL` and `VIEW_INDEX`. The panel contains LR, official base, refined output, target, confidence, residual, both error maps, and two Fourier spectra.

In [ ]:
from IPython.display import Image as DisplayImage, display

VIEW_MODEL = MODELS_TO_RUN[0]
VIEW_INDEX = 0
image_dir = RUN_ROOT / VIEW_MODEL / "sn_hfr" / "paired_images" / "test"
overviews = sorted(image_dir.glob("*_overview.png"))
if not overviews:
    raise FileNotFoundError(f"No paired diagnostics in {image_dir}")
VIEW_INDEX = max(0, min(VIEW_INDEX, len(overviews) - 1))
print("Showing:", overviews[VIEW_INDEX])
display(DisplayImage(filename=str(overviews[VIEW_INDEX])))

## 14. Archive checkpoints, metrics, and diagnostics

The archive excludes cloned official repositories and the attached NPZ dataset. No files are deleted.

In [ ]:
archive_base = Path("/kaggle/working") / f"sn_hfr_paired_runs_{time.strftime('%Y%m%d_%H%M%S')}"
archive_path = shutil.make_archive(str(archive_base), "gztar", root_dir=RUN_ROOT)
print("Archive:", archive_path)
print("Manifest:", LOCAL_MANIFEST)
print("No source, dataset, checkpoint, or personal file was deleted.")